In [ ]:
#@title Bibliotecas e Ambiente de Execução
import os
import io
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

In [ ]:
# parametros iniciais
smoth_DTG = 95
smoth_TGA = 95

In [ ]:
# Obtendo a lista de arquivos .txt
data_path = 'data'
files = [os.path.join(data_path, file_) for file_ in os.listdir(data_path) if os.path.isfile(os.path.join(data_path, file_))]
files_txt = [arquivo for arquivo in files if arquivo.lower().endswith('.txt')]
files_txt.sort()
print(len(files_txt))

In [ ]:
# definiçao da funcao de primaria

def dtg(df):

  # ajuste de massa zero
  df['mg'] = df['mg'] - df['mg'].min()

  # Obter massa porcentual
  massa_i = df['mg'].iloc[0]
  df['massa (%)'] = (df['mg'] / massa_i) * 100

  # Agora que as colunas são numéricas, calcule a derivada
  df['DTG'] = df['mg'].diff() / df['sec'].diff()

  # suavizaçao
  df['DTG_smoth'] = df['DTG'].rolling(window=int(smoth_DTG), center=True, min_periods=1).mean()
  df['massa (%)_smoth'] = df['massa (%)'].rolling(window=int(smoth_TGA), center=True, min_periods=1).mean()

  df_ = df.rename(columns={'sec': 'tempo (s)','C': 'temperatura (C)','mg': 'massa (mg)','DTG': 'DTG (%/°C)','uV': 'DTA (uV)','DTG_smoth': 'DTG_smoth (%/°C)'})

  # Verifique o resultado
  #print(df.head())
  #print(df.info())
  return df_

In [ ]:
# definiçao da funcao de grafico

def grafico_dtg(df, eixo_x='temperatura (C)', eixo_y1 ='massa (mg)', eixo_y2 = 'DTG_smoth (%/°C)', material = 'nome_do_material'):
  # Garante que as colunas são numéricas (caso ainda não sejam)
  df[eixo_y1] = pd.to_numeric(df[eixo_y1], errors='coerce')
  df[eixo_y2] = pd.to_numeric(df[eixo_y2], errors='coerce')
  df[eixo_x] = pd.to_numeric(df[eixo_x], errors='coerce')

  # Cria a figura e o primeiro eixo (ax1)
  fig, ax1 = plt.subplots(figsize=(12, 6))

  # Cor para o primeiro eixo e gráfico (massa)
  cor_eixo_y1 = 'tab:blue'
  ax1.set_xlabel(eixo_x)
  ax1.set_ylabel(eixo_y1, color=cor_eixo_y1)
  ax1.plot(df[eixo_x], df[eixo_y1], color=cor_eixo_y1, label=eixo_y1)
  ax1.tick_params(axis='y', labelcolor=cor_eixo_y1)
  ax1.grid(True, linestyle='--', alpha=0.6)

  # Cria um segundo eixo (ax2) que compartilha o mesmo eixo X
  ax2 = ax1.twinx()

  # Cor para o segundo eixo e gráfico (derivada)
  cor_eixo_y2 = 'tab:red'
  ax2.set_ylabel(eixo_y2, color=cor_eixo_y2)
  ax2.plot(df[eixo_x], df[eixo_y2], color=cor_eixo_y2, label=eixo_y2)
  ax2.tick_params(axis='y', labelcolor=cor_eixo_y2)

  # Adiciona um título geral e exibe o gráfico
  plt.title(f'MATERIAL: {material}')
  fig.tight_layout() # Ajusta o layout para evitar sobreposição
  plt.show()

In [ ]:
# definiçao da funcao secundaria

def open_file(path_):
  try:
      material_name = path_.split('/')[-1].split('.')[0]

      # abre o docuemnto txt como csv
      df = pd.read_csv(path_, sep="\t", skiprows=2)
      df_ = df.iloc[60:].copy()

      for col in df_.columns:
          try:
              df_[col] = df_[col].str.replace(',', '.')
              df_[col] = pd.to_numeric(df_[col])
          except AttributeError:
              continue

      #execuçao da funçao primaria
      DF = dtg(df_)

      # salva o dataframe em xlsx
      xlsx_path = path_.replace('.txt', '.xlsx')
      DF.to_excel(xlsx_path, index=False)

      dtg_dta = grafico_dtg(DF, eixo_x='temperatura (C)', eixo_y1='DTG_smoth (%/°C)', eixo_y2='DTA (uV)', material = material_name)
      tga_dtg = grafico_dtg(DF, eixo_x='temperatura (C)', eixo_y1='massa (mg)', eixo_y2='DTG_smoth (%/°C)', material = material_name)
      tga_dta = grafico_dtg(DF, eixo_x='temperatura (C)', eixo_y1='massa (mg)', eixo_y2='DTA (uV)', material = material_name)

      return DF
  except FileNotFoundError:
      print(f"O arquivo '{path_}' não foi encontrado.")
      return None
  except Exception as e:
      print(f"Ocorreu um erro ao abrir o arquivo: {e}")
      return None

In [ ]:
# execuçao da funçao

for path_ in files_txt:
  print(path_)
  df = open_file(path_)
  if df is not None:
    print(df.head())
  else:
    print(f"Falha ao processar: {path_}")